In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

assert torch.cuda.is_available(), "GPU is not enabled!"

In [ ]:
import os
import glob
import shutil

matches = glob.glob(
    "/kaggle/input/**/run_all_kaggle.sh",
    recursive=True
)

print("Found project:", matches)

assert matches, "Could not find run_all_kaggle.sh in /kaggle/input"

source_dir = os.path.dirname(matches[0])
project_dir = "/kaggle/working/AIR_Phase3_completed"

shutil.rmtree(project_dir, ignore_errors=True)
shutil.copytree(source_dir, project_dir)

os.chdir(project_dir)

print("Source:", source_dir)
print("Working directory:", os.getcwd())
print("\nProject files:")
print(os.listdir("."))

In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable,
    "-c",
    "import numpy, pandas, scipy; "
    "print('NumPy:', numpy.__version__); "
    "print('Pandas:', pandas.__version__); "
    "print('SciPy:', scipy.__version__); "
    "print('Scientific stack: OK')"
], check=True)

In [ ]:
!pip install -q --upgrade-strategy only-if-needed \
    "faiss-cpu<1.11" \
    "sentence-transformers[image]>=5.4" \
    "transformers>=4.57" \
    "qwen-vl-utils>=0.0.14" \
    "tabulate"

In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable,
    "-c",
    """
import torch
import torchvision
import numpy
import pandas
import scipy
import pyarrow
import transformers
import sentence_transformers
import faiss
import qwen_vl_utils

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("SciPy:", scipy.__version__)
print("PyArrow:", pyarrow.__version__)
print("Transformers:", transformers.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("FAISS: OK")
print("Qwen VL Utils: OK")
print("\\nENVIRONMENT IS READY")
"""
], check=True)

In [ ]:
!python check_kaggle_gpu.py

In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable,
    "-c",
    """
import os
import pandas as pd

files = [
    "data/catalog_subset.parquet",
    "data/queries.jsonl",
    "data/qrels.jsonl",
    "run_all_kaggle.sh",
    "src/build_index.py",
    "src/retrieve.py",
    "src/rerank.py",
    "src/evaluate.py",
    "src/generate_answer.py",
    "src/tests.py",
]

for path in files:
    print(("OK     " if os.path.exists(path) else "MISSING"), path)

catalog = pd.read_parquet("data/catalog_subset.parquet")
print("\\nProducts:", len(catalog))
"""
], check=True)

In [ ]:
!DENSE_BATCH_SIZE=1 \
SPARSE_BATCH_SIZE=8 \
RERANK_BATCH_SIZE=16 \
bash run_all_kaggle.sh

In [ ]:
import os

expected = [
    "artifacts/embeddings/product_dense.npy",
    "artifacts/embeddings/product_ids.txt",
    "artifacts/embeddings/product_sparse.jsonl",
    "artifacts/index/dense_index.faiss",
    "artifacts/index/sparse_index.npz",
    "artifacts/index/index_manifest.json",
    "outputs/retrieval_results.jsonl",
    "outputs/reranked_results.jsonl",
    "data/qrels.jsonl",
    "reports/evaluation_metrics.csv",
    "reports/evaluation_report.md",
    "reports/error_analysis.md",
    "outputs/final_answers.jsonl",
]

all_ok = True

for path in expected:
    exists = os.path.exists(path)
    print("OK     " if exists else "MISSING", path)
    all_ok = all_ok and exists

print("\nAll final outputs available:", all_ok)

In [ ]:
import subprocess
import sys

for stage in range(6):
    print("\n" + "=" * 60)
    print(f"FINAL ACCEPTANCE TEST — STAGE {stage}")
    print("=" * 60)

    result = subprocess.run(
        [sys.executable, "src/tests.py", "--stage", str(stage)]
    )

    if result.returncode != 0:
        print(f"\n❌ Stage {stage} has failed checks.")
        break
else:
    print("\n✅ ALL STAGE 0–5 ACCEPTANCE TESTS PASSED")

In [ ]:
import shutil
import os

final_path = "/kaggle/working/AIR_Phase3_FINAL_COMPLETED"

shutil.make_archive(
    final_path,
    "zip",
    "/kaggle/working/AIR_Phase3_completed"
)

print("Final project created:")
print(final_path + ".zip")

## STAGE 0

In [ ]:
from pathlib import Path
import os, glob, shutil, subprocess, sys

# Find the completed project. This notebook can run:
# 1) after the main notebook in the same Kaggle session, or
# 2) in a new notebook with the completed project attached as a Kaggle Dataset.
direct_candidates = [
    Path.cwd(),
    Path("/kaggle/working/AIR_Phase3_completed"),
    Path("/content/AIR_Phase3_completed"),
]
found_roots = [
    Path(p).parent.parent
    for p in glob.glob("/kaggle/working/**/src/tests.py", recursive=True)
]
found_roots += [
    Path(p).parent.parent
    for p in glob.glob("/content/**/src/tests.py", recursive=True)
]

PROJECT_DIR = next(
    (p.resolve() for p in direct_candidates + found_roots if (p / "src/tests.py").exists()),
    None,
)

# If the project is attached under /kaggle/input, copy it to a writable folder.
if PROJECT_DIR is None:
    input_matches = glob.glob("/kaggle/input/**/src/tests.py", recursive=True)
    if input_matches:
        source_root = Path(input_matches[0]).parent.parent
        writable_root = Path("/kaggle/working/AIR_Phase3_completed")
        if writable_root.exists():
            shutil.rmtree(writable_root)
        shutil.copytree(source_root, writable_root)
        PROJECT_DIR = writable_root.resolve()

if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Could not find the completed project. Run the main notebook first, "
        "or attach the completed project folder as a Kaggle Dataset."
    )

os.chdir(PROJECT_DIR)
print("Project:", PROJECT_DIR)

# Default mode: reuse completed outputs and focus on checks and plots.
REBUILD_STAGE1 = False
RERUN_STAGE2 = False
RERUN_STAGE3 = False
RERUN_STAGE4 = True
RERUN_STAGE5 = False
RUN_STRONGER_LLM = True
RUN_OPTIONAL_LORA = True

subprocess.run([sys.executable, "src/tests.py", "--stage", "0"], check=True)


## STAGE 1

In [ ]:
import importlib
import platform

required = [
    "numpy", "pandas", "scipy", "pyarrow", "torch",
    "torchvision", "transformers", "sentence_transformers", "faiss"
]
print("Python:", platform.python_version())
for name in required:
    try:
        module = importlib.import_module(name)
        print(f"{name:24s} OK  {getattr(module, '__version__', '')}")
    except Exception as exc:
        print(f"{name:24s} MISSING/ERROR: {exc}")

print("\nThe main notebook should install the runtime. This notebook does not change packages.")

In [ ]:
import json, math, collections, shutil, subprocess, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown, JSON

DATA_DIR = Path("data")
ARTIFACT_DIR = Path("artifacts")
OUTPUT_DIR = Path("outputs")
REPORT_DIR = Path("reports")
FIG_DIR = REPORT_DIR / "notebook_figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

def load_jsonl(path):
    path = Path(path)
    return [json.loads(line) for line in path.open(encoding="utf-8") if line.strip()]

def resolve_image(path):
    if not path:
        return None
    p = Path(str(path))
    checks = [p, PROJECT_DIR / p, DATA_DIR / p]
    for candidate in checks:
        if candidate.exists():
            return candidate
    return PROJECT_DIR / p

print("Reusable inspection mode is ready.")

In [ ]:
catalog_path = DATA_DIR / "catalog_subset.parquet"
if not catalog_path.exists():
    raise FileNotFoundError(catalog_path)

catalog = pd.read_parquet(catalog_path)
required_columns = {"product_id", "title", "product_text", "image_path"}
missing_columns = required_columns - set(catalog.columns)
if missing_columns:
    raise ValueError(f"Missing catalog columns: {sorted(missing_columns)}")

image_ok = catalog["image_path"].map(
    lambda p: bool(p) and resolve_image(p).exists()
)
print("Products:", len(catalog))
print("Unique product IDs:", catalog["product_id"].nunique())
print("Existing images:", int(image_ok.sum()), "/", len(catalog))
print("Product types:", catalog["product_type"].nunique())

type_counts = catalog["product_type"].fillna("UNKNOWN").value_counts().head(15)
ax = type_counts.sort_values().plot(
    kind="barh", figsize=(8, 5), title="Top product types in the 2,000-product catalog"
)
ax.set_xlabel("Product count")
plt.tight_layout()
plt.savefig(FIG_DIR / "product_type_distribution.png", dpi=150)
plt.show()

display(catalog.head(5))

In [ ]:
DENSE_MODEL = "Qwen/Qwen3-VL-Embedding-2B"
SPARSE_MODEL = "prithivida/Splade_PP_en_v2"
DENSE_BATCH_SIZE = 1
SPARSE_BATCH_SIZE = 8

stage1_required = [
    ARTIFACT_DIR / "embeddings/product_dense.npy",
    ARTIFACT_DIR / "embeddings/product_ids.txt",
    ARTIFACT_DIR / "embeddings/product_sparse.jsonl",
    ARTIFACT_DIR / "index/dense_index.faiss",
    ARTIFACT_DIR / "index/sparse_index.npz",
    ARTIFACT_DIR / "index/index_manifest.json",
]

if REBUILD_STAGE1 or not all(p.exists() for p in stage1_required):
    cmd = [
        sys.executable, "src/build_index.py",
        "--device", "auto",
        "--dense_batch_size", str(DENSE_BATCH_SIZE),
        "--sparse_batch_size", str(SPARSE_BATCH_SIZE),
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("Stage 1 artifacts already exist; model inference is skipped.")

In [ ]:
dense_path = ARTIFACT_DIR / "embeddings/product_dense.npy"
product_dense = np.load(dense_path, mmap_mode="r")
dense_norms = np.linalg.norm(np.asarray(product_dense[: min(500, len(product_dense))]), axis=1)

print("Dense matrix shape:", product_dense.shape)
print("Dense dtype:", product_dense.dtype)
print("Sample norm range:", float(dense_norms.min()), "..", float(dense_norms.max()))

In [ ]:
product_ids = [
    line.strip()
    for line in (ARTIFACT_DIR / "embeddings/product_ids.txt").open(encoding="utf-8")
    if line.strip()
]
assert len(product_ids) == len(product_dense)
assert len(set(product_ids)) == len(product_ids)

id_to_row = {pid: i for i, pid in enumerate(product_ids)}
catalog_by_id = catalog.set_index("product_id", drop=False)

plt.figure(figsize=(6, 3))
plt.hist(dense_norms, bins=20)
plt.title("Dense-vector norm check")
plt.xlabel("L2 norm")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(FIG_DIR / "dense_norms.png", dpi=150)
plt.show()

In [ ]:
sparse_records = load_jsonl(ARTIFACT_DIR / "embeddings/product_sparse.jsonl")
nnz = np.array([len(r.get("indices", [])) for r in sparse_records])

print("Sparse product records:", len(sparse_records))
print("Average retained dimensions:", round(float(nnz.mean()), 2))
print("Min / max retained dimensions:", int(nnz.min()), int(nnz.max()))
display(pd.DataFrame({
    "product_id": [r["product_id"] for r in sparse_records[:5]],
    "nnz": [len(r["indices"]) for r in sparse_records[:5]],
}))

In [ ]:
import scipy.sparse as sp

sparse_index = sp.load_npz(ARTIFACT_DIR / "index/sparse_index.npz")
print("Sparse index shape:", sparse_index.shape)
print("Sparse index nnz:", sparse_index.nnz)
print("Average nnz/product:", round(sparse_index.nnz / sparse_index.shape[0], 2))

plt.figure(figsize=(6, 3))
plt.hist(nnz, bins=20)
plt.title("SPLADE retained dimensions per product")
plt.xlabel("Non-zero dimensions")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(FIG_DIR / "sparse_nnz.png", dpi=150)
plt.show()

In [ ]:
import faiss

dense_index = faiss.read_index(str(ARTIFACT_DIR / "index/dense_index.faiss"))
print("FAISS index type:", type(dense_index).__name__)
print("FAISS products:", dense_index.ntotal)
print("FAISS dimension:", dense_index.d)
assert dense_index.ntotal == len(product_ids)

In [ ]:
manifest = json.loads(
    (ARTIFACT_DIR / "index/index_manifest.json").read_text(encoding="utf-8")
)
display(pd.DataFrame(
    [{"setting": key, "value": value} for key, value in manifest.items()]
))

In [ ]:
SANITY_TOP_K = 5

queries = load_jsonl(DATA_DIR / "queries.jsonl")
queries_by_id = {q["query_id"]: q for q in queries}
retrieval_records = load_jsonl(OUTPUT_DIR / "retrieval_results.jsonl")
retrieval_by_id = {r["query_id"]: r for r in retrieval_records}

print("Queries:", len(queries))
print("Retrieval records:", len(retrieval_records))
print("Top-k used for notebook plots:", SANITY_TOP_K)

In [ ]:
summary_path = REPORT_DIR / "indexing_summary.md"
if summary_path.exists():
    display(Markdown(summary_path.read_text(encoding="utf-8")))
else:
    print("Indexing summary is missing.")

In [ ]:
subprocess.run([sys.executable, "src/tests.py", "--stage", "1"], check=True)

In [ ]:
def short_text(value, limit=48):
    text = "" if value is None else str(value)
    return text if len(text) <= limit else text[: limit - 1] + "…"

def show_hits(record, score_key="prefusion_score", top_n=5, title=None):
    hits = record["results"][:top_n]
    fig, axes = plt.subplots(1, len(hits), figsize=(3.0 * len(hits), 3.8))
    if len(hits) == 1:
        axes = [axes]

    for ax, hit in zip(axes, hits):
        pid = hit["product_id"]
        row = catalog_by_id.loc[pid]
        image_path = resolve_image(row["image_path"])
        try:
            ax.imshow(Image.open(image_path).convert("RGB"))
        except Exception:
            ax.text(0.5, 0.5, "Image unavailable", ha="center", va="center")
        ax.axis("off")
        score = hit.get(score_key)
        score_text = "n/a" if score is None else f"{float(score):.3f}"
        ax.set_title(
            f"#{hit['rank']} {short_text(row['title'])}\n{score_key}={score_text}",
            fontsize=8,
        )

    q = queries_by_id.get(record["query_id"], {})
    heading = title or q.get("query_text") or q.get("query_image_path") or record["query_id"]
    fig.suptitle(f"{record['query_id']}: {heading}", fontsize=11)
    plt.tight_layout()
    return fig

In [ ]:
representatives = ["q001", "q011", "q016"]
for qid in representatives:
    fig = show_hits(retrieval_by_id[qid], "prefusion_score", SANITY_TOP_K)
    fig.savefig(FIG_DIR / f"{qid}_retrieval.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
qid = "q003"
record = retrieval_by_id[qid]

dense_hits = sorted(
    record["results"],
    key=lambda h: float(h.get("dense_score") or -1e9),
    reverse=True,
)
sparse_hits = sorted(
    [h for h in record["results"] if h.get("sparse_score") is not None],
    key=lambda h: float(h["sparse_score"]),
    reverse=True,
)

dense_record = {**record, "results": dense_hits}
sparse_record = {**record, "results": sparse_hits}

fig = show_hits(dense_record, "dense_score", SANITY_TOP_K, "Dense-only ordering")
fig.savefig(FIG_DIR / "q003_dense_order.png", dpi=150, bbox_inches="tight")
plt.show()

fig = show_hits(sparse_record, "sparse_score", SANITY_TOP_K, "Sparse-only ordering")
fig.savefig(FIG_DIR / "q003_sparse_order.png", dpi=150, bbox_inches="tight")
plt.show()

## STAGE 2 & 3

In [ ]:
os.chdir(PROJECT_DIR)
print("Stage 2/3 project directory:", Path.cwd())

In [ ]:
print("Using the environment prepared by the main notebook.")
print("No package installation is performed here.")

import torch, transformers, sentence_transformers
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Transformers:", transformers.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)

In [ ]:
print("Writable project:", os.access(PROJECT_DIR, os.W_OK))

stage23_files = [
    DATA_DIR / "catalog_subset.parquet",
    DATA_DIR / "queries.jsonl",
    ARTIFACT_DIR / "index/dense_index.faiss",
    ARTIFACT_DIR / "index/sparse_index.npz",
    OUTPUT_DIR / "retrieval_results.jsonl",
    OUTPUT_DIR / "reranked_results.jsonl",
]
for path in stage23_files:
    print(("OK     " if path.exists() else "MISSING"), path)


In [ ]:
TOP_K_DENSE = 100
TOP_K_SPARSE = 100
W_DENSE = 0.55
W_SPARSE = 0.45
RERANK_TOP_N = 100
RERANK_BATCH_SIZE = 16

print({
    "top_k_dense": TOP_K_DENSE,
    "top_k_sparse": TOP_K_SPARSE,
    "hybrid_weights": (W_DENSE, W_SPARSE),
    "rerank_top_n": RERANK_TOP_N,
})


catalog = pd.read_parquet(DATA_DIR / "catalog_subset.parquet")
catalog_by_id = catalog.set_index("product_id", drop=False)
queries = load_jsonl(DATA_DIR / "queries.jsonl")
queries_by_id = {q["query_id"]: q for q in queries}

if not (OUTPUT_DIR / "retrieval_results.jsonl").exists():
    RERUN_STAGE2 = True
if not (OUTPUT_DIR / "reranked_results.jsonl").exists():
    RERUN_STAGE3 = True

print("Catalog:", len(catalog))
print("Queries:", len(queries))

print("Dense query encoder:", manifest["dense_model"])
print("Input policy:", manifest["dense_embedding_policy"])
print("Image+text inputs are encoded as one multimodal vector.")
print("Sparse query encoder:", manifest["sparse_model"])
print("Sparse vectors are used for text and image+text queries.")
print("Image-only queries use the dense multimodal score.")
image_queries = [q for q in queries if q.get("query_image_path")]
missing_query_images = [
    q["query_id"]
    for q in image_queries
    if not resolve_image(q["query_image_path"]).exists()
]
print("Image-bearing queries:", len(image_queries))
print("Missing query images:", missing_query_images)
assert not missing_query_images

In [ ]:
query_table = pd.DataFrame(queries)[
    ["query_id", "query_type", "query_text", "query_image_path"]
]
display(query_table)

counts = query_table["query_type"].value_counts()
ax = counts.plot(kind="bar", figsize=(6, 3), title="Query types")
ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig(FIG_DIR / "query_type_counts.png", dpi=150)
plt.show()

In [ ]:
retrieval_records = load_jsonl(OUTPUT_DIR / "retrieval_results.jsonl")
source_rows = []
for record in retrieval_records:
    for hit in record["results"]:
        source = "+".join(hit.get("source", []))
        source_rows.append((record["query_type"], source))

source_df = pd.DataFrame(source_rows, columns=["query_type", "source"])
display(pd.crosstab(source_df["query_type"], source_df["source"]))

if RERUN_STAGE2:
    cmd = [
        sys.executable, "src/retrieve.py",
        "--device", "auto",
        "--top_k_dense", str(TOP_K_DENSE),
        "--top_k_sparse", str(TOP_K_SPARSE),
        "--prefusion", "weighted",
        "--w_dense", str(W_DENSE),
        "--w_sparse", str(W_SPARSE),
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("Using existing Stage 2 retrieval output.")

retrieval_records = load_jsonl(OUTPUT_DIR / "retrieval_results.jsonl")
retrieval_by_id = {r["query_id"]: r for r in retrieval_records}
print("Retrieval records:", len(retrieval_records))

In [ ]:
subprocess.run([sys.executable, "src/tests.py", "--stage", "2"], check=True)

In [ ]:
RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
print("Reranker:", RERANKER_MODEL)
print("The implemented reranker is text-based and keeps visual evidence through score fusion.")

In [ ]:
from src.common import product_rerank_text

sample_row = next(catalog.itertuples())
sample_text = product_rerank_text(sample_row, max_chars=500)
print(sample_text)

if RERUN_STAGE3:
    cmd = [
        sys.executable, "src/rerank.py",
        "--device", "cuda" if torch.cuda.is_available() else "cpu",
        "--batch_size", str(RERANK_BATCH_SIZE),
        "--rerank_top_n", str(RERANK_TOP_N),
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("Using existing Stage 3 reranking output.")

rerank_records = load_jsonl(OUTPUT_DIR / "reranked_results.jsonl")
rerank_by_id = {r["query_id"]: r for r in rerank_records}
print("Reranked records:", len(rerank_records))

In [ ]:
summary_path = REPORT_DIR / "reranking_summary.md"
if summary_path.exists():
    display(Markdown(summary_path.read_text(encoding="utf-8")))

subprocess.run([sys.executable, "src/tests.py", "--stage", "3"], check=True)

In [ ]:
# Save a lightweight copy of the notebook-visible reports and plots.
export_dir = PROJECT_DIR / "notebook_exports"
export_dir.mkdir(exist_ok=True)
for path in FIG_DIR.glob("*.png"):
    shutil.copy2(path, export_dir / path.name)
print("Notebook figures copied to:", export_dir)

summary_rows = []
for record in retrieval_records:
    qid = record["query_id"]
    before = record["results"][0]["product_id"]
    after = rerank_by_id[qid]["results"][0]["product_id"]
    summary_rows.append({
        "query_id": qid,
        "query_type": record["query_type"],
        "pool_size": len(record["results"]),
        "top1_before": before,
        "top1_after": after,
        "top1_changed": before != after,
    })

summary_df = pd.DataFrame(summary_rows)
print("Queries:", len(summary_df))
print("Pool-size range:", summary_df["pool_size"].min(), "..", summary_df["pool_size"].max())
print("Top-1 changes:", int(summary_df["top1_changed"].sum()), "/", len(summary_df))
display(summary_df)


In [ ]:
selected = ["q001", "q011", "q016", "q003"]

for qid in selected:
    before = retrieval_by_id[qid]["results"][:10]
    after = rerank_by_id[qid]["results"][:10]

    before_map = {h["product_id"]: h for h in before}
    common_ids = [h["product_id"] for h in after if h["product_id"] in before_map][:8]
    labels = [str(i + 1) for i in range(len(common_ids))]
    pre_scores = [before_map[pid]["prefusion_score"] for pid in common_ids]
    final_scores = [
        next(h["final_score"] for h in after if h["product_id"] == pid)
        for pid in common_ids
    ]

    x = np.arange(len(common_ids))
    width = 0.38
    plt.figure(figsize=(8, 3.5))
    plt.bar(x - width/2, pre_scores, width, label="Stage 2 prefusion")
    plt.bar(x + width/2, final_scores, width, label="Stage 3 final")
    plt.xticks(x, labels)
    plt.xlabel("Products in Stage 3 order")
    plt.ylabel("Score (different scales)")
    plt.title(f"{qid}: before vs after reranking")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{qid}_before_after_scores.png", dpi=150)
    plt.show()

    fig = show_hits(rerank_by_id[qid], "final_score", 5, "Final reranked products")
    fig.savefig(FIG_DIR / f"{qid}_reranked_products.png", dpi=150, bbox_inches="tight")
    plt.show()


In [ ]:
checks = []

def check(name, condition):
    checks.append({"check": name, "passed": bool(condition)})

check("20 retrieval records", len(retrieval_records) == 20)
check("20 rerank records", len(rerank_records) == 20)
check("Unique retrieval query IDs", len({r["query_id"] for r in retrieval_records}) == 20)
check("Unique rerank query IDs", len({r["query_id"] for r in rerank_records}) == 20)

catalog_ids = set(catalog["product_id"])
for r in rerank_records:
    retrieval_ids = {h["product_id"] for h in retrieval_by_id[r["query_id"]]["results"]}
    rerank_ids = {h["product_id"] for h in r["results"]}
    check(f"{r['query_id']} rerank subset", rerank_ids.issubset(retrieval_ids))
    check(f"{r['query_id']} products exist", rerank_ids.issubset(catalog_ids))
    check(f"{r['query_id']} rank order", [h["rank"] for h in r["results"]] == list(range(1, len(r["results"]) + 1)))

check_df = pd.DataFrame(checks)
display(check_df)
print("Passed:", int(check_df["passed"].sum()), "/", len(check_df))
assert check_df["passed"].all()

## STAGE 4

In [ ]:
queries = load_jsonl(DATA_DIR / "queries.jsonl")
qrels_path = DATA_DIR / "qrels.jsonl"

if not qrels_path.exists():
    raise FileNotFoundError("data/qrels.jsonl is missing.")
qrels = load_jsonl(qrels_path)
counts = collections.Counter(row["query_id"] for row in qrels)

print("Queries:", len(queries))
print("Qrels:", len(qrels))
print("Covered queries:", len(counts), "/", len(queries))
print("Judgments/query range:", min(counts.values()), "..", max(counts.values()))

display(pd.DataFrame(qrels).head(10))

In [ ]:
pool_path = DATA_DIR / "qrels_pool.csv"
if pool_path.exists():
    pool = pd.read_csv(pool_path)
    print("Judgment-pool rows:", len(pool))
    display(pool.head(10))
else:
    subprocess.run(
        [sys.executable, "src/build_judgment_pool.py", "--pool_out", str(pool_path), "--per_run", "10"],
        check=True,
    )
    pool = pd.read_csv(pool_path)
    display(pool.head(10))

In [ ]:
reviewed_pool = DATA_DIR / "qrels_pool_reviewed.csv"
if not qrels_path.exists() and reviewed_pool.exists():
    subprocess.run([
        sys.executable, "src/build_judgment_pool.py",
        "--finalize", str(reviewed_pool),
        "--qrels_out", str(qrels_path),
    ], check=True)
else:
    print("Final qrels already exist; no manual-edit step is needed in this inspection notebook.")

In [ ]:
if RERUN_STAGE4 or not (REPORT_DIR / "evaluation_metrics.csv").exists():
    subprocess.run([
        sys.executable, "src/evaluate.py",
        "--queries", "data/queries.jsonl",
        "--qrels", "data/qrels.jsonl",
        "--retrieval", "outputs/retrieval_results.jsonl",
        "--reranked", "outputs/reranked_results.jsonl",
        "--out", "reports/evaluation_metrics.csv",
        "--report", "reports/evaluation_report.md",
        "--error_analysis", "reports/error_analysis.md",
    ], check=True)
else:
    print("Using existing Stage 4 reports.")

In [ ]:
display(Markdown((REPORT_DIR / "evaluation_report.md").read_text(encoding="utf-8")))

metrics = pd.read_csv(REPORT_DIR / "evaluation_metrics.csv")
overall = metrics[metrics["query_type"] == "all"].copy()
plot_cols = ["precision_at_5", "recall_at_20", "map", "ndcg_at_10"]
plot_df = overall.set_index("run_name")[plot_cols]

ax = plot_df.plot(kind="bar", figsize=(10, 4))
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("Overall retrieval and reranking metrics")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(FIG_DIR / "overall_metrics.png", dpi=150)
plt.show()

display(overall)

In [ ]:
display(Markdown((REPORT_DIR / "error_analysis.md").read_text(encoding="utf-8")))

subprocess.run([sys.executable, "src/tests.py", "--stage", "4"], check=True)

## STAGE 5

In [ ]:
os.chdir(PROJECT_DIR)
print("Stage 5 project:", PROJECT_DIR)
print("The project is already available; no upload or extraction is needed.")


In [ ]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

LLM_MODEL_BASELINE = "Qwen/Qwen2.5-1.5B-Instruct"
LLM_TOP_K = 8
LLM_MAX_NEW_TOKENS = 700
LLM_REPAIR_ATTEMPTS = 2

## A. Student baseline: Qwen2.5-1.5B

In [ ]:
answers_path = OUTPUT_DIR / "final_answers.jsonl"
summary_path = REPORT_DIR / "llm_output_summary.md"

if RERUN_STAGE5 or not answers_path.exists():
    cmd = [
        sys.executable, "src/generate_answer.py",
        "--model", LLM_MODEL_BASELINE,
        "--top_k", str(LLM_TOP_K),
        "--max_new_tokens", str(LLM_MAX_NEW_TOKENS),
        "--repair_attempts", str(LLM_REPAIR_ATTEMPTS),
        "--device", "cuda" if torch.cuda.is_available() else "cpu",
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("Using the existing Stage 5 output.")

subprocess.run([sys.executable, "src/tests.py", "--stage", "5"], check=True)


## B. Stronger reference: Qwen2.5-3B


In [ ]:
LLM_MODEL_STRONGER = "Qwen/Qwen2.5-3B-Instruct"

if RUN_STRONGER_LLM:
    stronger_out = OUTPUT_DIR / "final_answers.local_qwen3b.jsonl"
    stronger_summary = REPORT_DIR / "llm_output_summary.local_qwen3b.md"
    subprocess.run([
        sys.executable, "src/generate_answer.py",
        "--model", LLM_MODEL_STRONGER,
        "--top_k", str(LLM_TOP_K),
        "--max_new_tokens", str(LLM_MAX_NEW_TOKENS),
        "--repair_attempts", str(LLM_REPAIR_ATTEMPTS),
        "--device", "cuda" if torch.cuda.is_available() else "cpu",
        "--out", str(stronger_out),
        "--summary", str(stronger_summary),
    ], check=True)
else:
    print("Stronger 3B run skipped. Set RUN_STRONGER_LLM=True to run it.")


In [ ]:
answers = load_jsonl(OUTPUT_DIR / "final_answers.jsonl")
print("Final answers:", len(answers))
display(JSON(answers[0]))

decision_counts = pd.Series(
    [a.get("decision", "unknown") for a in answers]
).value_counts()
ax = decision_counts.plot(kind="bar", figsize=(7, 3), title="Stage 5 decision distribution")
ax.set_ylabel("Queries")
plt.tight_layout()
plt.savefig(FIG_DIR / "stage5_decisions.png", dpi=150)
plt.show()

display(Markdown((REPORT_DIR / "llm_output_summary.md").read_text(encoding="utf-8")))

In [ ]:
comparison = [{
    "model": LLM_MODEL_BASELINE,
    "answers": len(answers),
    "schema_valid": len(answers),
    "output": str(OUTPUT_DIR / "final_answers.jsonl"),
}]
stronger_path = OUTPUT_DIR / "final_answers.local_qwen3b.jsonl"
if stronger_path.exists():
    stronger_answers = load_jsonl(stronger_path)
    comparison.append({
        "model": LLM_MODEL_STRONGER,
        "answers": len(stronger_answers),
        "schema_valid": len(stronger_answers),
        "output": str(stronger_path),
    })

comparison_df = pd.DataFrame(comparison)
display(comparison_df)
comparison_df.to_csv(REPORT_DIR / "stage5_model_comparison.csv", index=False)

In [ ]:
archive_base = PROJECT_DIR / "notebook_exports"
archive_path = shutil.make_archive(
    str(PROJECT_DIR / "AIR_Phase3_notebook_visuals"),
    "zip",
    root_dir=PROJECT_DIR,
    base_dir="notebook_exports",
)
print("Notebook visual bundle:", archive_path)
print("All required project outputs remain in the main project folder.")